# Segmentation des accidents par profil

### *Objectif* :
Regrouper les accidents en clusters homogènes selon leurs attributs (météo, type de route, heure, gravité, etc.). Cela fait ressortir des typologies d’accidents (ex. « accidents nocturnes sous pluie » vs « accidents de jour en centre-ville »)

In [ ]:
from pyspark.sql import SparkSession
import os
import pandas as pd
spark = SparkSession.builder\
    .master("local[*]")\
    .appName("Accident")\
    .config("spark.driver.memory", "4g") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
path = "file://" + os.path.abspath(os.path.join(os.getcwd(), "../../data/processed_accidents.parquet"))


df = spark.read.parquet(path)
df.show(5, truncate=False)


25/06/19 10:09:05 WARN Utils: Your hostname, bg-ubuntu resolves to a loopback address: 127.0.1.1; using 192.168.1.47 instead (on interface wlp0s20f3)
25/06/19 10:09:05 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/06/19 10:09:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+--------+-------------------+-------------------+---------+-----------+------------------+---------------+-----------+-----+--------------+-------------+-----------+------------+--------------+--------------+---------------+-----------------+-----------------+-------+----+--------+--------+--------+-------+-------+----------+-------+----+---------------+--------------+------------+--------------+--------------+-----------------+---------------------+----------------------+----------------------+-------------------------+-----------------------------+-------------------------+----------------------+-------------+------------+----------------------+----------------------+-------------------------+-----------------------------+-------------------------+----------------------+---------------+--------------------+
|Severity|Start_Time         |End_Time           |Start_Lat|Start_Lng  |Distance(mi)      |City           |County     |State|Temperature(F)|Wind_Chill(F)|Humidity(%)|Pressure(in)

+--------+-------------------+-------------------+---------+-------------------+------------+----------------+--------+-----+--------------+-------------+-----------+------------+--------------+--------------+---------------+-----------------+--------------------+-------+----+--------+--------+--------+-------+-------+----------+-------+----+---------------+--------------+------------+--------------+--------------+-----------------+---------------------+----------------------+----------------------+-------------------------+-----------------------------+-------------------------+----------------------+-------------+------------+----------------------+----------------------+-------------------------+-----------------------------+-------------------------+----------------------+---------------+--------------------+
|Severity|         Start_Time|           End_Time|Start_Lat|          Start_Lng|Distance(mi)|            City|  County|State|Temperature(F)|Wind_Chill(F)|Humidity(%)|Pressure(

25/06/19 15:18:57 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$driverEndpoint(BlockManagerMasterEndpoint.scala:123)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.isExecutorAlive$lzycompute$1(BlockManagerMasterEndpoint.scala:688)
	at org.apache.spark.storage.BlockManagerMasterE

In [10]:
index_to_state = df['City','City_indexed'].distinct()
index_to_state = index_to_state.toPandas().set_index('City_indexed')['City'].to_dict()
print("State index mapping:", index_to_state)


State index mapping: {12.0: 'San Diego', 91.0: 'Stockton', 136.0: 'Downey', 4982.0: 'South Rockwood', 1650.0: 'Henrietta', 641.0: 'Dillon', 423.0: 'Woodland Hills', 362.0: 'Leesburg', 456.0: 'Palmetto', 975.0: 'Richfield', 1501.0: 'Kernersville', 873.0: 'Spanish Fork', 5866.0: 'Glenshaw', 2516.0: 'Wallace', 3449.0: 'South Bend', 2278.0: 'Pinon Hills', 1646.0: 'Blythe', 4486.0: 'Sweet Springs', 5901.0: 'Grayson', 6304.0: 'Okawville', 3498.0: 'Heppner', 5432.0: 'Wills Point', 4146.0: 'Avondale Estates', 6176.0: 'Guin', 4133.0: 'Raeford', 9627.0: 'Washingtonville', 4301.0: 'Midland City', 5079.0: 'Waldron', 7023.0: 'Upton', 10004.0: 'Grove Hill', 5178.0: 'Plattsburgh', 7244.0: 'Edgar', 9206.0: 'St James City', 8503.0: 'Dane', 4128.0: 'Marlboro', 6820.0: 'Dardanelle', 6855.0: 'Schuyler', 11186.0: 'Monclova', 9880.0: 'Thornwood', 6989.0: 'Fort Klamath', 6114.0: 'Wilkesboro', 8208.0: 'Mayview', 13532.0: 'St Benedict', 8500.0: 'Colma', 11174.0: 'McNeil', 8096.0: 'Modale', 8645.0: 'Fyffe', 106

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.pipeline import Pipeline
from pyspark.ml.feature import VectorAssembler
# Standardisation
features = ['Temperature(F)', 'Sunrise_Sunset_indexed'
]

X = df[features].dropna()
assembler = VectorAssembler(
    inputCols=features,
    outputCol="features"
)
X_pandas = X.toPandas()
X_vector = assembler.transform(X_pandas)

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_vector)

# Réduction de dimension
pca = PCA(n_components=10)
X_pca = pca.fit_transform(X_scaled)

# Clustering
kmeans = KMeans(n_clusters=5, random_state=42)
clusters = kmeans.fit_predict(X_pca)

# Ajout des clusters au dataframe
df['Cluster'] = clusters

# Visualisation
import matplotlib.pyplot as plt
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=clusters, cmap='viridis')
plt.show()
